In [1]:
def score_transaction(transaction_df):
    
    # Apply the saved preprocessing pipeline
    transaction_processed = loaded_preprocessor.transform(transaction_df)
    
    # Generate fraud probability
    risk_score = loaded_xgb_model.predict_proba(
        transaction_processed
    )[:, 1][0]
    
    # Assign risk level
    if risk_score < 0.30:
        risk_level = "Low"
    elif risk_score < 0.70:
        risk_level = "Medium"
    else:
        risk_level = "High"
    
    # Binary fraud prediction
    fraud_prediction = int(risk_score >= 0.70)
    
    return {
        "fraud_risk_score": round(float(risk_score), 6),
        "risk_level": risk_level,
        "fraud_prediction": fraud_prediction
    }

In [8]:
print("X_test" in globals())
print("X_test_processed" in globals())
print("loaded_preprocessor" in globals())
print("loaded_xgb_model" in globals())

False
False
False
False


In [9]:
print("final_output" in globals())
print("production_output" in globals())
print("loaded_preprocessor" in globals())
print("loaded_xgb_model" in globals())

False
False
False
False


In [10]:
import os

print(os.listdir())

['data_inspection.ipynb', 'fraud_preprocessor.pkl', 'fraud_xgb_model.pkl', 'ml_analysis.ipynb', 'score_prediction.ipynb']


In [11]:
import joblib

loaded_preprocessor = joblib.load("fraud_preprocessor.pkl")
loaded_xgb_model = joblib.load("fraud_xgb_model.pkl")

print("Preprocessor loaded:", type(loaded_preprocessor))
print("Model loaded:", type(loaded_xgb_model))

Preprocessor loaded: <class 'sklearn.compose._column_transformer.ColumnTransformer'>
Model loaded: <class 'xgboost.sklearn.XGBClassifier'>


In [12]:
print(loaded_preprocessor.feature_names_in_)

['merchant' 'category' 'amt' 'gender' 'lat' 'long' 'city_pop' 'job'
 'merch_lat' 'merch_long' 'transaction_hour' 'transaction_day'
 'transaction_month' 'transaction_dayofweek' 'is_weekend' 'age'
 'distance_km']


In [13]:
# Testing the scoring function with a sample transaction 
test_transaction = {
    "merchant": "fraud_Ferry, Lynch and Kautzer",
    "category": "misc_net",
    "amt": 6.64,
    "gender": "F",
    "lat": 40.5553,
    "long": -77.4001,
    "city_pop": 1909,
    "job": "Mining engineer",
    "merch_lat": 41.405953,
    "merch_long": -77.922023,
    "transaction_hour": 3,
    "transaction_day": 29,
    "transaction_month": 4,
    "transaction_dayofweek": 2,
    "is_weekend": 0,
    "age": 66,
    "distance_km": 104.241872
}

In [15]:
import joblib
import pandas as pd

# Load saved model and preprocessing pipeline
preprocessor = joblib.load("fraud_preprocessor.pkl")
model = joblib.load("fraud_xgb_model.pkl")


def score_transaction(transaction):
    """
    Score a single transaction for fraud risk.

    transaction should be a dictionary containing
    the same raw features used during model training.
    """

    transaction_df = pd.DataFrame([transaction])

    # Apply the exact preprocessing used during training
    transaction_processed = preprocessor.transform(transaction_df)

    # Generate fraud probability
    risk_score = model.predict_proba(
        transaction_processed
    )[0, 1]

    # Assign risk level
    if risk_score < 0.30:
        risk_level = "Low"
    elif risk_score < 0.70:
        risk_level = "Medium"
    else:
        risk_level = "High"

    # Final fraud prediction
    fraud_prediction = int(risk_score >= 0.70)

    return {
        "fraud_risk_score": round(float(risk_score), 6),
        "risk_level": risk_level,
        "fraud_prediction": fraud_prediction
    }

In [16]:
result = score_transaction(test_transaction)

result

{'fraud_risk_score': 0.001771, 'risk_level': 'Low', 'fraud_prediction': 0}

In [17]:
fraud_test_transaction = {
    "merchant": "fraud_Predovic Inc",
    "category": "shopping_net",
    "amt": 1115.50,
    "gender": "M",
    "lat": 39.2853,
    "long": -77.6462,
    "city_pop": 1518,
    "job": "Psychotherapist",
    "merch_lat": 39.1180,
    "merch_long": -76.9920,
    "transaction_hour": 23,
    "transaction_day": 1,
    "transaction_month": 1,
    "transaction_dayofweek": 4,
    "is_weekend": 0,
    "age": 37,
    "distance_km": 56.49
}

score_transaction(fraud_test_transaction)

{'fraud_risk_score': 0.994597, 'risk_level': 'High', 'fraud_prediction': 1}